In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
# Data Ingest:
expedia_data = pd.read_csv("Data_by_site_cleaned/Site_5.csv")
# expedia_data = pd.read_csv("Data_by_site_cleaned/Site_14.csv")
# expedia_data = pd.read_csv("Data_by_site_cleaned/Site_15.csv")

In [2]:
# Combined Data:
site_5 = pd.read_csv("Data_by_site_cleaned/Site_5.csv")
site_14 = pd.read_csv("Data_by_site_cleaned/Site_14.csv")
site_15 = pd.read_csv("Data_by_site_cleaned/Site_15.csv")

site_5["srch_id"] = "5_" + site_5["srch_id"].astype(str)
site_14["srch_id"] = "14_" + site_14["srch_id"].astype(str)
site_15["srch_id"] = "15_" + site_15["srch_id"].astype(str)

expedia_data = pd.concat(
    [site_5, site_14, site_15],
    ignore_index=True
)

In [2]:
# Optional interaction features

expedia_data["price_x_stars"] = (
    expedia_data["price_usd"] * expedia_data["prop_starrating"]
)

expedia_data["price_x_location"] = (
    expedia_data["price_usd"] * expedia_data["prop_location_score1"]
)

expedia_data["price_x_histprice"] = (
    expedia_data["price_usd"] * expedia_data["prop_log_historical_price"]
)

expedia_data["promo_x_price"] = (
    expedia_data["promotion_flag"] * expedia_data["price_usd"]
)

expedia_data["promo_x_stars"] = (
    expedia_data["promotion_flag"] * expedia_data["prop_starrating"]
)

expedia_data["price_x_trip"] = (
    expedia_data["price_usd"] * expedia_data["srch_length_of_stay"]
)

expedia_data["price_x_window"] = (
    expedia_data["price_usd"] * expedia_data["srch_booking_window"]
)

expedia_data["price_x_occupancy"] = (
    expedia_data["price_usd"] * (expedia_data["srch_adults_count"] + expedia_data["srch_children_count"])
)

expedia_data["stars_x_trip"] = (
    expedia_data["prop_starrating"] * expedia_data["srch_length_of_stay"]
)

In [3]:
# Base feature_list:
feature_list = [
    "price_usd",
    "prop_starrating",
    "prop_location_score1",
    "prop_log_historical_price",
    "promotion_flag"
]

In [4]:
# Base features + interaction terms:
feature_list = [
    # base features
    "price_usd",
    "prop_starrating",
    "prop_location_score1",
    "prop_log_historical_price",
    "promotion_flag",

    # interactions
    "price_x_stars",
    "price_x_location",
    "price_x_histprice",
    "promo_x_price",
    "promo_x_stars",
    
    # additional interactions
    "price_x_trip",
    "price_x_window",
    "price_x_occupancy",
    "stars_x_trip"
]

In [5]:
def Estimate_MNL_Expedia(train_data,
                         feature_list,
                         learning_rate=1e-4,
                         epochs=1000,
                         batch_size=256,
                         l2_reg=1e-3):
    """
    Data Transformations:
        Pad + Mask: Make tensors have the same shape for each observation by adding zeros, then mask to remove them mathematically.
        Feature Standardization: Features are on different scales (stars and price).
    
    Args:
        train_data (_type_): _description_
        feature_list (_type_): _description_
        learning_rate (_type_, optional): _description_. Defaults to 1e-4.
        epochs (int, optional): _description_. Defaults to 1000.
        batch_size (int, optional): _description_. Defaults to 256.
        l2_reg (_type_, optional): _description_. Defaults to 1e-3.

    Returns:
        _type_: _description_
    """
    

    import numpy as np
    import tensorflow as tf

    num_features = len(feature_list)

    # ---------------------------------------
    # Build vertically differentiated sets
    # ---------------------------------------

    offered = []
    purchased = []

    for srch_id, g in train_data.groupby("srch_id"):

        g = g.sort_values("position")

        if g.click_bool.sum() == 0:
            continue

        deepest_click = g.loc[g.click_bool == 1, "position"].max()
        g = g[g.position <= deepest_click]

        X = g[feature_list].values.astype(np.float32)

        if g.booking_bool.sum() > 0:
            chosen = g.loc[g.booking_bool == 1, feature_list].iloc[0].values.astype(np.float32)
        else:
            chosen = g.loc[g.click_bool == 1, feature_list].iloc[0].values.astype(np.float32)

        offered.append(X)
        purchased.append(chosen)

    # ---------------------------------------
    # PAD + MASK
    # ---------------------------------------

    max_J = max(x.shape[0] for x in offered)

    X_pad = np.zeros((len(offered), max_J, num_features), dtype=np.float32)
    mask = np.zeros((len(offered), max_J), dtype=np.float32)
    y_pad = np.zeros((len(offered), num_features), dtype=np.float32)

    for i, (X, y) in enumerate(zip(offered, purchased)):
        J = X.shape[0]
        X_pad[i, :J, :] = X
        mask[i, :J] = 1
        y_pad[i] = y

    # ---------------------------------------
    # FEATURE STANDARDIZATION
    # ---------------------------------------

    mean = X_pad.mean(axis=(0, 1), keepdims=True)
    std = X_pad.std(axis=(0, 1), keepdims=True) + 1e-8

    X_pad = (X_pad - mean) / std
    y_pad = (y_pad - mean.squeeze()) / std.squeeze()

    # ---------------------------------------
    # TF SETUP
    # ---------------------------------------

    tf.compat.v1.disable_eager_execution()

    offer_mnl = tf.compat.v1.placeholder(tf.float32, [None, max_J, num_features])
    purchase_mnl = tf.compat.v1.placeholder(tf.float32, [None, num_features])
    mask_ph = tf.compat.v1.placeholder(tf.float32, [None, max_J])

    W = tf.Variable(tf.random.normal([num_features], stddev=0.01))

    # ---------------------------------------
    # UTILITIES
    # ---------------------------------------

    utilities = tf.reduce_sum(offer_mnl * W, axis=2)

    # -------------------------------
    # STABLE LOG-SUM-EXP (FIXED)
    # -------------------------------

    u_max = tf.reduce_max(utilities, axis=1, keepdims=True)

    exp_u = tf.exp(utilities - u_max) * mask_ph

    log_denom = u_max[:, 0] + tf.math.log(tf.reduce_sum(exp_u, axis=1) + 1e-8)

    # ---------------------------------------
    # CHOSEN UTILITY
    # ---------------------------------------

    chosen_utility = tf.reduce_sum(purchase_mnl * W, axis=1)

    # ---------------------------------------
    # NEG LOG LIKELIHOOD (IMPORTANT FIX: MEAN)
    # ---------------------------------------

    nll = tf.reduce_mean(log_denom - chosen_utility)

    # L2 regularization
    l2_penalty = l2_reg * tf.reduce_sum(tf.square(W))

    neg_log_like = nll + l2_penalty

    optimizer = tf.compat.v1.train.AdamOptimizer(learning_rate).minimize(neg_log_like)

    init = tf.compat.v1.global_variables_initializer()

    # ---------------------------------------
    # TRAINING LOOP
    # ---------------------------------------

    n = X_pad.shape[0]

    with tf.compat.v1.Session() as sess:

        sess.run(init)

        for epoch in range(epochs):

            total_ll = 0

            for i in range(0, n, batch_size):

                X_batch = X_pad[i:i+batch_size]
                y_batch = y_pad[i:i+batch_size]
                m_batch = mask[i:i+batch_size]

                _, loss = sess.run(
                    [optimizer, neg_log_like],
                    feed_dict={
                        offer_mnl: X_batch,
                        purchase_mnl: y_batch,
                        mask_ph: m_batch
                    }
                )

                total_ll += loss

            if epoch % 10 == 0:
                print(f"Epoch {epoch}, NegLL: {total_ll:.4f}")

        W_final = sess.run(W)

    return W_final

In [6]:
# 
W = Estimate_MNL_Expedia(
    train_data=expedia_data,
    feature_list=feature_list,
    learning_rate=1e-5,
    epochs=1000
)

print(W)



Epoch 0, NegLL: 455.5464
Epoch 10, NegLL: 452.8465
Epoch 20, NegLL: 451.6725
Epoch 30, NegLL: 450.8750
Epoch 40, NegLL: 450.2278
Epoch 50, NegLL: 449.6856
Epoch 60, NegLL: 449.2299
Epoch 70, NegLL: 448.8480
Epoch 80, NegLL: 448.5281
Epoch 90, NegLL: 448.2608
Epoch 100, NegLL: 448.0366
Epoch 110, NegLL: 447.8472
Epoch 120, NegLL: 447.6857
Epoch 130, NegLL: 447.5468
Epoch 140, NegLL: 447.4258
Epoch 150, NegLL: 447.3194
Epoch 160, NegLL: 447.2249
Epoch 170, NegLL: 447.1399
Epoch 180, NegLL: 447.0630
Epoch 190, NegLL: 446.9928
Epoch 200, NegLL: 446.9286
Epoch 210, NegLL: 446.8690
Epoch 220, NegLL: 446.8139
Epoch 230, NegLL: 446.7626
Epoch 240, NegLL: 446.7148
Epoch 250, NegLL: 446.6697
Epoch 260, NegLL: 446.6280
Epoch 270, NegLL: 446.5883
Epoch 280, NegLL: 446.5512
Epoch 290, NegLL: 446.5161
Epoch 300, NegLL: 446.4831
Epoch 310, NegLL: 446.4517
Epoch 320, NegLL: 446.4223
Epoch 330, NegLL: 446.3944
Epoch 340, NegLL: 446.3678
Epoch 350, NegLL: 446.3427
Epoch 360, NegLL: 446.3188
Epoch 370,

## Dataset 01: Site_05

### Full data:
Epoch 990, NegLL: 447.9326 (with all features present).
- -3.5555115e-01  
- 3.6327514e-01  
- 1.4376697e-01  
- 5.6846201e-01
- 5.7792995e-02
- -4.7276852e-05
- -2.1185194e-06 
- 4.5843964e-05
- 1.5762824e-05
- -1.3486863e-04 
- -2.8771706e-07 
- -1.3479240e-04

### Relevant feature set results (srch variables removed):
Epoch 990, NegLL: 447.1022
- -0.4012041 (price)
- 0.357356 (stars)
- 0.14595902 (location score)
- 0.8871471 (log historical price)
- 0.05695665 (promotion flag)

### Relevant features with interaction terms:
Epoch 990, NegLL: 446.5753
- -0.22115314 (price)
- 0.3286871 (stars)
- 0.26520056 (location score) 
- 0.8680734 (log historical price)
- 0.01672585 (promotion flag)
- -0.00491137 (price x stars)
- -0.20868069 (price x location)
- 0.01719261 (price x histprice)
- -0.05598945 (promo x price)
- 0.08545977 (promo x stars)

### Relevant features + interaction terms including srch variables:
Epoch 990, NegLL: 445.8345
- -0.2748329 (price)
- 0.30780596 (stars)
- 0.2662657 (location score)
- 0.9171545 (log historical price)  
- 0.02777621 (promotion flag)
- -0.07128749 (price x stars)
- -0.2032002 (price x location)
- -0.02233708 (price x histprice)
- -0.06098413 (promo x price)
- 0.0766582 (promo x stars)
- 0.16948111 (price x trip length)
- 0.04964715 (price x window)
- -0.03431497 (price x occupancy) 
- 0.08179977 (stars x trip)

## Dataset 02: Site_14

- -0.1474525 (price)
- 0.25434268 (stars)
- 0.25039425 (location score)
- 0.32637572 (log historical price)
- 0.07244041 (promotion flag)
- -0.05351232 (price x stars)
- -0.08640814 (price x location)
- -0.03466194 (price x histprice)
- -0.07642625 (promo x price)
- 0.07615843 (promo x stars)
- -0.00126741 (price x trip length)
- 0.02381122 (price x window)
- -0.05389581 (price x occupancy)
- 0.23428467 (stars x trip)

## Dataset 03: Site_15

- -0.07342983 (price)
- 0.0612151 (stars)
- 0.17759103 (location score)
- 0.20836082 (log historical price)
- 0.03420797 (promotion flag)
- -0.01826278 (price x stars)
- -0.03822072 (price x location)
- -0.02753216 (price x histprice)
- -0.02658776 (promo x price)
- 0.04684472 (promo x stars)
- -0.01064833 (price x trip length)
- -0.04795501 (price x window)
- -0.05523824 (price x occupancy)
- 0.08833089 (stars x trip)

## Combined Data:

- -0.29476374 (price)
- 0.30840743 (stars)
- 0.28591445 (location score)
- 0.9213034 (log historical price)
- 0.02862852 (promotion flag)
- -0.06208701 (price x star)
- -0.21032625 (price x location)
- -0.02326492 (price x histprice)
- -0.06409068 (promo x price)
- 0.08060129 (promo x stars)
- 0.17053276 (price x trip length)
- 0.04154778 (price x window)
- -0.01765287 (price occupancy)
- 0.06864065 (stars x trip)

## Stop Position IP Fit

This section fits the first-trigger stop-position integer program using:
- `expedia_data` from this notebook,
- `feature_list` from this notebook,
- the fitted MNL coefficient vector `W` from the prior estimation cell.

Run the next cell after `W` is available. It reports in-sample fit metrics and the learned monotone threshold sequence.

In [ ]:
# Main IP formulation: learn monotone reservation thresholds by customer type.
import os
import sys
import importlib
import numpy as np

# Prerequisite: this notebook must have already estimated MNL coefficients into W.
if "W" not in globals():
    raise RuntimeError(
        "Missing W. Run the MNL estimation cell first so the optimization can use fitted coefficients."
    )

# Make returns_theoretical importable for either notebook-dir or workspace-root execution.
candidate_roots = [
    os.getcwd(),
    os.path.abspath("."),
    os.path.abspath(".."),
]
for root in candidate_roots:
    if os.path.isdir(os.path.join(root, "returns_theoretical")) and root not in sys.path:
        sys.path.append(root)

import returns_theoretical.stop_position_ip as stop_ip
importlib.reload(stop_ip)

fit_stop_ip_from_notebook_outputs = stop_ip.fit_stop_ip_from_notebook_outputs
build_session_examples_from_mnl = stop_ip.build_session_examples_from_mnl

solver_backend = "gurobi" if (stop_ip.gp is not None and stop_ip.GRB is not None) else "scipy.milp"
print("Solver backend:", solver_backend)

# Use same columns/order as the estimated MNL coefficient vector W.
beta_hat = np.array(W).reshape(-1)

# Explicitly choose number of latent customer types in the main IP.
n_customer_types = 3

epsilon = 1e-6

# ------------------------------
# Pre-solve verification checks
# ------------------------------
missing_features = [c for c in feature_list if c not in expedia_data.columns]
if missing_features:
    raise ValueError(f"Missing feature columns in expedia_data: {missing_features}")

if len(beta_hat) != len(feature_list):
    raise ValueError(
        f"beta_hat length ({len(beta_hat)}) does not match feature_list length ({len(feature_list)})."
    )

# Verify click filtering set sizes before solving.
session_has_click = expedia_data.groupby("srch_id")["click_bool"].sum() > 0
sessions_with_click = set(session_has_click[session_has_click].index.astype(str))
sessions_without_click = set(session_has_click[~session_has_click].index.astype(str))

print("Sessions with >=1 click:", len(sessions_with_click))
print("Sessions with 0 clicks:", len(sessions_without_click))
print("Feature / beta alignment check: PASS")

ip_result, train_examples, mean_vec, std_vec = fit_stop_ip_from_notebook_outputs(
    train_data=expedia_data,
    feature_list=feature_list,
    beta_hat=beta_hat,
    n_customer_types=n_customer_types,
    epsilon=epsilon,
    session_col="srch_id",
    position_col="position",
    click_col="click_bool",
)

# Verify no-click sessions were excluded in the fitted examples.
example_session_ids = {str(ex.session_id) for ex in train_examples}
leaked_no_click_sessions = sorted(example_session_ids.intersection(sessions_without_click))
if leaked_no_click_sessions:
    raise RuntimeError(
        "Found sessions without clicks in train_examples; filtering failed. "
        f"Examples leaked: {leaked_no_click_sessions[:10]}"
    )

expected_example_sessions = len(sessions_with_click)
if len(train_examples) != expected_example_sessions:
    print(
        "Warning: train_examples count does not exactly match clicked-session count. "
        f"examples={len(train_examples)}, clicked_sessions={expected_example_sessions}"
    )
else:
    print("No-click filtering check: PASS")

# Avg predicted rank (1-indexed) and avg true rank (1-indexed).
pred_ranks = [ip_result.predicted_stop_idx[ex.session_id] + 1 for ex in train_examples]
true_ranks = [ex.true_stop_idx + 1 for ex in train_examples]

# Compact assignment summary: type index -> session count and percentage.
assignment_values = list(ip_result.assigned_type_idx.values())
unassigned_count = int(sum(1 for t in assignment_values if t < 0))
assigned_values = [t for t in assignment_values if t >= 0]

if assigned_values:
    type_ids, type_counts = np.unique(np.array(assigned_values, dtype=int), return_counts=True)
    assignment_summary = {int(t): int(c) for t, c in zip(type_ids, type_counts)}
    assignment_pct = {
        int(t): round(100.0 * int(c) / len(train_examples), 2)
        for t, c in zip(type_ids, type_counts)
    }
else:
    assignment_summary = {}
    assignment_pct = {}

unassigned_pct = round(100.0 * unassigned_count / len(train_examples), 2)

print("=== Stop Position IP (In-Sample) ===")
print("Sessions used:", len(train_examples))
print("Customer types (K):", n_customer_types)
print("Epsilon:", epsilon)
print("Objective (explained sessions):", ip_result.objective_hits)
print("Hit Rate:", round(ip_result.hit_rate, 4))
print("Avg Predicted Rank:", round(float(np.mean(pred_ranks)), 3))
print("Avg True Rank:", round(float(np.mean(true_ranks)), 3))
print("Type assignment counts:", assignment_summary)
print("Type assignment percentages (% of sessions):", assignment_pct)
print("Unassigned sessions:", unassigned_count)
print("Unassigned percentage:", unassigned_pct)
print("Threshold matrix shape [N, K]:", ip_result.thresholds.shape)
print("Threshold preview (first 10 positions):")
print(np.round(ip_result.thresholds[:10, :], 6))

# Optional OOS wiring: uncomment after defining a held-out DataFrame test_data.
# test_examples = build_session_examples_from_mnl(
#     data=test_data,
#     feature_list=feature_list,
#     beta_hat=beta_hat,
#     mean=mean_vec,
#     std=std_vec,
#     session_col="srch_id",
#     position_col="position",
#     click_col="click_bool",
# )
#
# def predict_with_thresholds(examples, thresholds):
#     preds = {}
#     for ex in examples:
#         j_q = len(ex.weights)
#         pred = -1
#         cum_w = np.cumsum(ex.weights)
#         for k in range(thresholds.shape[1]):
#             t_q = ex.true_stop_idx
#             pre_ok = True
#             for p in range(t_q):
#                 if cum_w[p] > thresholds[p, k] - epsilon:
#                     pre_ok = False
#                     break
#             if pre_ok and cum_w[t_q] >= thresholds[t_q, k]:
#                 pred = t_q
#                 break
#         preds[ex.session_id] = pred
#     return preds
#
# test_preds = predict_with_thresholds(test_examples, ip_result.thresholds)
# test_hit = np.mean([test_preds[ex.session_id] == ex.true_stop_idx for ex in test_examples])
# test_avg_rank = np.mean([test_preds[ex.session_id] + 1 for ex in test_examples if test_preds[ex.session_id] >= 0])
# print("OOS Hit Rate:", round(float(test_hit), 4))
# print("OOS Avg Predicted Rank:", round(float(test_avg_rank), 3))

Solver backend: scipy.milp
=== Stop Position IP (In-Sample) ===
Sessions used: 63301
Objective (correct stops): 7669.0
Hit Rate: 0.1212
Avg Predicted Rank: 1.0
Avg True Rank: 9.821
Learned thresholds r_p (first 15): [1.36e-04 2.40e-05 2.40e-05 2.40e-05 0.00e+00 0.00e+00 0.00e+00 0.00e+00
 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00 0.00e+00]


In [8]:
expedia_data = pd.read_csv("Data_by_site_cleaned/Site_5.csv")

In [ ]:
# Find srch sessions whose clicked reservation is only the first reservation.
clicked = expedia_data.loc[expedia_data["click_bool"] == 1, ["srch_id", "position"]].copy()

session_click_summary = (
    clicked.groupby("srch_id")
    .agg(
        click_count=("position", "size"),
        clicked_positions=("position", lambda positions: list(positions)),
        min_clicked_position=("position", "min"),
        max_clicked_position=("position", "max"),
    )
    .reset_index()
)

# Sessions where the only clicked item is position 1.
first_reservation_only_sessions = session_click_summary.loc[
    (session_click_summary["click_count"] == 1)
    & (session_click_summary["min_clicked_position"] == 1)
].copy()

print("Total sessions with at least one click:", len(session_click_summary))
print("Sessions whose only click is the first reservation:", len(first_reservation_only_sessions))
print(first_reservation_only_sessions.head(20))

Total sessions with at least one click: 63301
Sessions whose only click is the first reservation: 7629
     srch_id  click_count clicked_positions  min_clicked_position  \
2         12            1               [1]                     1   
5         61            1               [1]                     1   
15       140            1               [1]                     1   
18       152            1               [1]                     1   
21       162            1               [1]                     1   
30       221            1               [1]                     1   
40       361            1               [1]                     1   
46       419            1               [1]                     1   
50       466            1               [1]                     1   
51       483            1               [1]                     1   
59       562            1               [1]                     1   
75       767            1               [1]                     1   


: 